In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import requests
import tensorflow as tf
from transformers import GPT2Tokenizer, TFGPT2LMHeadModel
import PyPDF2
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

In [23]:
# Prepare training texts
'''train_texts = []
for record in data['info']:
    title = record['title']
    for req in record['requests']:
        for resp in record['responses']:  # keep all responses
            train_texts.append(f"User: {req}\nBot: {resp}")'''

pdf_file = "vertrag.pdf"
train_texts = ""
'''
with open(pdf_file, "rb") as f:
    reader = PyPDF2.PdfReader(f)
    for page in reader.pages:
        train_texts += page.extract_text() + "\n"'''


# -------------------------
# 1. Load CSV
# -------------------------
csv_file = "dataset.csv"
df = pd.read_csv(csv_file)

# Remove rows with missing important values
df = df.dropna(subset=["prompt", "answer"])

# -------------------------
# 2. Build training texts
# -------------------------
train_texts = []

for _, row in df.iterrows():
    prompt = str(row["prompt"]).strip()
    answer = str(row["answer"]).strip()

    choice_a = str(row["A"]).strip() if pd.notna(row["A"]) else ""
    choice_b = str(row["B"]).strip() if pd.notna(row["B"]) else ""
    choice_c = str(row["C"]).strip() if pd.notna(row["C"]) else ""
    choice_d = str(row["D"]).strip() if pd.notna(row["D"]) else ""
    choice_e = str(row["E"]).strip() if pd.notna(row["E"]) else ""

    text = f"""User: {prompt}
Choices:
A: {choice_a}
B: {choice_b}
C: {choice_c}
D: {choice_d}
E: {choice_e}
Bot: The correct answer is {answer}"""

    train_texts.append(text)

# Load GPT-2 tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Tokenize all texts
inputs = tokenizer(train_texts, return_tensors="tf", padding=True, truncation=True)

# Load GPT-2 model
model = TFGPT2LMHeadModel.from_pretrained("gpt2")

# Convert to tf.data.Dataset
train_dataset = tf.data.Dataset.from_tensor_slices({
    "input_ids": inputs["input_ids"],
    "attention_mask": inputs["attention_mask"],
    "labels": inputs["input_ids"]
})

train_dataset = train_dataset.shuffle(100).batch(8)

# Compile model
optimizer = tf.keras.optimizers.Adam(learning_rate=5e-5,epsilon=1e-7)
model.compile(optimizer=optimizer)

# Train
model.fit(train_dataset, epochs=3)

All PyTorch model weights were used when initializing TFGPT2LMHeadModel.

All the weights of TFGPT2LMHeadModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFGPT2LMHeadModel for predictions without further training.


Epoch 1/3
Cause: for/else statement not yet supported
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Cause: for/else statement not yet supported
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert




KeyboardInterrupt: 

In [105]:
# Save model + tokenizer
model.save_pretrained("./my_model")
tokenizer.save_pretrained("./my_model")

print("Model saved!")

Model saved!


In [2]:
# Chat function
def chat_gpt(prompt,max_length=60):
    input_ids = tokenizer.encode(prompt, return_tensors="tf")
    
    # Generate response
    output_ids = model.generate(
        input_ids,
        max_length=max_length,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id
    )
    output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    # Extract only the bot's response
    if "Bot:" in output_text:
        return output_text.split("Bot:")[1].strip()
    else:
        return output_text

In [3]:
model = TFGPT2LMHeadModel.from_pretrained("./my_model")
tokenizer = GPT2Tokenizer.from_pretrained("./my_model")

print("Model loaded!")

All model checkpoint layers were used when initializing TFGPT2LMHeadModel.

All the layers of TFGPT2LMHeadModel were initialized from the model checkpoint at ./my_model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFGPT2LMHeadModel for predictions without further training.


Model loaded!


In [4]:
tokenizer.pad_token = tokenizer.eos_token

In [5]:
# 2. Load new data
df = pd.read_csv("tesla_dataset.csv").dropna(subset=["prompt", "answer"])

In [6]:
new_texts = []
for _, row in df.iterrows():
    text = f"User: {row['prompt'].strip()}\nBot: {row['answer'].strip()}"
    new_texts.append(text)


In [7]:
# 3. Tokenize
inputs = tokenizer(new_texts, return_tensors="tf", padding=True, 
                   truncation=True, max_length=128)

In [8]:
# 4. Build dataset
dataset = tf.data.Dataset.from_tensor_slices({
    "input_ids": inputs["input_ids"],
    "attention_mask": inputs["attention_mask"],
    "labels": inputs["input_ids"]
}).shuffle(500).batch(8).cache().prefetch(tf.data.AUTOTUNE)

In [71]:
# 5. Continue training (lower learning rate!)
optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5)  # lower than before
model.compile(optimizer=optimizer, jit_compile=True)
model.fit(dataset, epochs=10)

Epoch 1/10
4/4 [==============================] - 272s 35s/step - loss: 0.3946
Epoch 2/10
4/4 [==============================] - 12s 3s/step - loss: 0.3621
Epoch 3/10
4/4 [==============================] - 13s 3s/step - loss: 0.3034
Epoch 4/10
4/4 [==============================] - 13s 3s/step - loss: 0.2691
Epoch 5/10
4/4 [==============================] - 13s 3s/step - loss: 0.2541
Epoch 6/10
4/4 [==============================] - 13s 3s/step - loss: 0.2372
Epoch 7/10
4/4 [==============================] - 13s 3s/step - loss: 0.2297
Epoch 8/10
4/4 [==============================] - 13s 3s/step - loss: 0.2173
Epoch 9/10
4/4 [==============================] - 13s 3s/step - loss: 0.1988
Epoch 10/10
4/4 [==============================] - 13s 3s/step - loss: 0.2134


In [72]:
# Save model + tokenizer
model.save_pretrained("./my_model")
tokenizer.save_pretrained("./my_model")

print("Model saved!")

Model saved!


In [9]:
# 3. Search Tool
# -------------------------
from ddgs import DDGS
def search_web(query):
    results = []
    with DDGS() as ddgs:
        for r in ddgs.text(query, max_results=5):
            results.append({
                "body": r["body"],
            })
    return results

In [10]:
#Train the model to detect when to use the pre-trained data or show option
knn = KNeighborsClassifier(n_neighbors=5)

In [11]:
data_train = pd.read_csv("decision_dataset.csv")

In [12]:
# After loading data
from sklearn.feature_extraction.text import TfidfVectorizer
X = data_train[['question']]
Y = data_train[['decision']]

# ADD THIS — convert text to numbers
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(data_train['question'])

x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.25, random_state=0)

knn.fit(x_train, y_train)

# Test it
def needs_search(user_input):
    X_new = vectorizer.transform([user_input])
    return knn.predict(X_new)[0] == "SEARCH"

C:\Jupyter\tf_env\Lib\site-packages\sklearn\neighbors\_classification.py:243: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)


In [13]:
needs_search("Explain to me what is Tesla")

False

In [14]:
knn.score(x_test,y_test)

0.92

In [15]:
def combine_search_results(query):
    texts = []
    for item in query:
        body = item['body']
        body = body.replace("...", "")
        body = body.replace("Missing:", "")
        body = body.replace("Show results with:", "")
        body = body.strip()
        texts.append(body)
    #combine
    combined = " | ".join(texts)
    return combined[:400]

In [24]:
def generate_response(full_input, context="", plan=None):
    prompt = build_prompt(full_input, context)
    
    input_ids = tokenizer.encode(
        prompt,
        return_tensors="tf",
        truncation=True,
        max_length=200
    )
    output_ids = model.generate(
        input_ids,
        max_length=input_ids.shape[1] + 80,
        do_sample=False,
        repetition_penalty=1.5,
        no_repeat_ngram_size=4,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        early_stopping=True,
        length_penalty=0.6,
        num_beams=4,
    )
    output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    if "Bot:" in output_text:
        response = output_text.split("Bot:")[-1].strip()
    else:
        response = output_text.strip()
    
    # Cleanups
    if "\n" in response:
        response = response.split("\n")[0].strip()
    if "User:" in response:
        response = response.split("User:")[0].strip()
    if len(response) > 200:
        response = response[:200].rsplit(' ', 1)[0] + "."
    
    return response

In [22]:
conversation_history = []

def build_prompt(user_input, context="", plan=None):
    global conversation_history

    system_message = (
        "You are a helpful AI assistant. "
        "Follow the plan before answering. "
        "Answer clearly and in full sentences. "
        "If you do not know, say you do not know."
    )

    history= [system_message]
    history = "\n".join(conversation_history)

    if history:
        history += "\n"

    if context:
        prompt = f"{history}Context: {context}\nUser: {user_input}\nBot:"
    else:
        prompt = f"{history}User: {user_input}\nBot:"

    return prompt

def store_turn(user_input, response):
    global conversation_history
    conversation_history.append(f"User: {user_input}")
    conversation_history.append(f"Bot: {response}")
    conversation_history = conversation_history[-10:]   # keep last turns only

In [27]:
current_plan = []
def make_plan(question):
    global current_plan
    q = question.lower().strip()

    if "build" in q and "chatbot" in q:
        current_plan = [
            "Understand the goal of the chatbot",
            "Identify the main components",
            "Suggest the first implementation step",
            "Explain the next step if needed"
        ]
    elif "compare" in q or "difference" in q:
        current_plan = [
            "Identify the two things to compare",
            "List the main differences",
            "Give a short conclusion"
        ]
    elif "who is" in q or "what is" in q or "when" in q or "where" in q:
        current_plan = [
            "Identify whether the question is factual",
            "Decide whether search is needed",
            "Answer clearly and briefly"
        ]
    else:
        current_plan = [
            "Understand the question",
            "Use memory or search if needed",
            "Give the best possible answer"
        ]

    return current_plan

In [32]:
def fix_bad_response(question, response):
    if not response.strip():
        return "I do not know."

    if len(response.split()) < 3:
        return "I do not have enough reliable information to answer that clearly."

    return response

In [33]:
def agent_loop(question):
    plan = make_plan(question)
    print("Plan:", plan)

    context = ""
    if needs_search(question):
        print("need Search")
        context = combine_search_results(search_web(question))

    response = generate_response(question, context=context, plan=plan)
    response = fix_bad_response(question, response)

    return response

In [34]:
while True:
    question = input("You: ").strip()

    if question.lower() == "exit":
        print("\nBot: Good Bye!")
        break

    response = agent_loop(question)
    print("Bot:", response)

    store_turn(question, response)

You:  hello


Plan: ['Understand the question', 'Use memory or search if needed', 'Give the best possible answer']
Bot: I do not have enough reliable information to answer that clearly.


You:  what is Tesla ?


Plan: ['Identify whether the question is factual', 'Decide whether search is needed', 'Answer clearly and briefly']
Bot: Tesla is a Serbian-American inventor and electrical engineer born in 1856.


You:  Who is Asad ?


...

Plan: ['Identify whether the question is factual', 'Decide whether search is needed', 'Answer clearly and briefly']
Bot: Azyz is the biggest Muslim in the world


You:  who is azyz hanachi ?


Plan: ['Identify whether the question is factual', 'Decide whether search is needed', 'Answer clearly and briefly']
Bot: I do not have enough reliable information to answer that clearly.


You:  exit



Bot: Good Bye!
